# FactoryVision — 세그멘테이션 데이터셋 구축 + YOLO11-seg 학습

**목적:** 공개 데이터셋의 사각형(bbox) 어노테이션을 SAM 프롬프트로 사용해
픽셀 단위 폴리곤 라벨로 승격시키고, 이를 학습해 결함의 **형상과 면적까지 계측**한다.

```
기존 bbox 라벨 ──프롬프트──▶ MobileSAM ──▶ 마스크 ──윤곽추출──▶ 폴리곤 라벨
                                                                    │
                                        표본 검수 ◀───────────────────┤
                                                                    ▼
                                                         yolo11n-seg 학습
```

**설계 근거:** 모델 예측 박스가 아니라 **정답(GT) 박스**를 SAM 프롬프트로 쓴다.
원본 어노테이션의 검증된 위치 정보를 그대로 보존하면서 형상 정보만 추가하기 위함이다.
(이 때문에 학습된 `best.pt`는 이 노트북에서 필요하지 않다.)

**실행 전:** `런타임 > 런타임 유형 변경 > T4 GPU` 선택

In [ ]:
# 1. 패키지 설치 + GPU 확인
%pip install -q ultralytics

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임을 GPU로 변경하세요!)")

## 2. 데이터셋 다운로드

검출 모델 학습에 썼던 것과 **동일한 PCB 결함 데이터셋**을 받는다.
Roboflow의 `Download Dataset > YOLOv11 > show download code`에서 나오는
**zip 링크**를 `DATASET_URL`에 넣는다.

In [ ]:
DATASET_URL = ""  # Roboflow zip 링크 (비워두면 이미 /content/dataset에 있는 이미지를 그대로 사용)

import zipfile
from pathlib import Path

if DATASET_URL:
    # universe.roboflow.com은 Cloudflare 봇 차단에 걸려 curl에 403(HTML)을 돌려준다.
    # 같은 경로를 app.roboflow.com으로 요청하면 zip이 정상적으로 내려온다.
    url = DATASET_URL.replace("universe.roboflow.com/ds/", "app.roboflow.com/ds/")
    if url != DATASET_URL:
        print("universe.roboflow.com → app.roboflow.com 으로 변경 (Cloudflare 차단 회피)")

    !curl -sSL "{url}" -o roboflow.zip

    # 다운로드 실패를 조용히 넘기지 않는다 — 오류 응답이 zip 이름으로 저장되는 일이 흔하다
    zp = Path("roboflow.zip")
    print(f"내려받은 크기: {zp.stat().st_size / 1e6:.1f} MB")
    if not zipfile.is_zipfile(zp):
        print("\n[응답 내용 앞부분]")
        print(zp.read_bytes()[:400].decode("utf-8", "replace"))
        raise SystemExit(
            "zip이 아닙니다 — 다운로드가 실패했습니다.\n"
            "'Just a moment' / 'Enable JavaScript'가 보이면 Cloudflare 차단,\n"
            "'expired' / 'Unauthorized'가 보이면 링크 만료입니다.\n"
            "→ Roboflow의 Versions 탭에서 YOLOv11 링크를 새로 발급받으세요."
        )
    with zipfile.ZipFile(zp) as z:
        z.extractall("/content/dataset")
    print("압축 해제 완료")
else:
    print("DATASET_URL이 비어 있습니다 — 기존 /content/dataset을 그대로 사용합니다.")

d = Path("/content/dataset")
imgs = [p for p in d.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
print(f"현재 /content/dataset 이미지 {len(imgs):,}장")

In [ ]:
# 데이터셋 점검 — 박스를 어디서 가져올지 여기서 결정한다
#
# 정상 경로: 데이터셋의 정답(GT) 박스를 SAM 프롬프트로 쓴다.
# 우회 경로: 어노테이션이 없는 export를 받은 경우, 학습된 best.pt의 예측 박스를 쓴다.
#           (mAP50 0.9842 모델이 자신이 학습한 이미지를 다시 보는 것이므로 신뢰할 만하다)
import random
from pathlib import Path

import yaml

ROOT = Path("/content/dataset")
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
VAL_RATIO = 0.2   # 검증 split이 없을 때 떼어낼 비율
SPLIT_SEED = 42   # 고정 시드 — 재실행해도 같은 split이 나온다

if not ROOT.exists() or not any(ROOT.iterdir()):
    raise SystemExit("압축 해제 결과가 비어 있습니다. DATASET_URL과 앞 셀의 curl 출력을 확인하세요.")

# 이미지를 먼저 모으고, 라벨은 있으면 붙인다 (labels 폴더가 아예 없어도 진행 가능)
found, annotated_total = {}, 0
for img_dir in sorted(ROOT.rglob("images")):
    if not img_dir.is_dir():
        continue
    lbl_dir = img_dir.parent / "labels"
    imgs = [p for p in sorted(img_dir.iterdir()) if p.suffix.lower() in IMG_EXT]
    if not imgs:
        continue
    pairs = [(p, lbl_dir / f"{p.stem}.txt" if lbl_dir.is_dir() else None) for p in imgs]
    annotated = sum(1 for _, lbl in pairs if lbl and lbl.exists() and lbl.stat().st_size > 0)
    annotated_total += annotated
    found[img_dir.parent.name] = pairs
    print(f"  {img_dir.parent.name:8} 이미지 {len(imgs):>6,}장 · 어노테이션 {annotated:>6,}개")

if not found:
    print("[구조 진단] images 폴더를 찾지 못했습니다. 실제 구조:")
    for p in sorted(ROOT.rglob("*")):
        if p.is_dir():
            files = [f for f in p.iterdir() if f.is_file()]
            print(f"  {p.relative_to(ROOT)}/  파일 {len(files)}개 "
                  f"{sorted({f.suffix.lower() for f in files})}")
    raise SystemExit("이미지를 찾지 못했습니다 — 위 구조를 확인하세요.")

BOX_SOURCE = "labels" if annotated_total else "model"
if BOX_SOURCE == "model":
    print("\n⚠️ 라벨이 비어 있습니다 — 데이터셋에 어노테이션이 없는 export입니다.")
    print("   우회 경로로 전환합니다: 학습된 best.pt의 예측 박스를 SAM 프롬프트로 사용합니다.")
    print("   (정상 데이터셋을 받으려면 Roboflow의 Versions 탭에서 YOLOv11 포맷으로 다시 받으세요)")

# 검증 split이 없으면 떼어낸다 (원본 폴더는 건드리지 않고 배정만 한다)
val_key = next((k for k in found if k in ("valid", "val")), None)
if val_key and any(k.startswith("train") for k in found):
    SPLITS = {"train": found[next(k for k in found if k.startswith("train"))],
              "valid": found[val_key]}
else:
    pool = [p for pairs in found.values() for p in pairs]
    random.Random(SPLIT_SEED).shuffle(pool)
    cut = int(len(pool) * (1 - VAL_RATIO))
    SPLITS = {"train": pool[:cut], "valid": pool[cut:]}
    print(f"\n⚠️ 검증 split이 없어 {int(VAL_RATIO * 100)}%를 분리했습니다 "
          f"(seed={SPLIT_SEED}) — 원본 학습과 split이 달라 지표를 1:1 비교할 수는 없습니다.")

# 클래스 정의 — labels 경로일 때만 여기서 정한다 (model 경로는 다음 셀에서 best.pt로부터 가져온다)
PCB_DEFECT_NAMES = ["missing_hole", "mouse_bite", "open_circuit", "short", "spur", "spurious_copper"]
CLASSES = []
if BOX_SOURCE == "labels":
    yaml_files = sorted(ROOT.rglob("data.yaml"), key=lambda p: len(p.parts))
    names = yaml.safe_load(yaml_files[0].read_text()).get("names") if yaml_files else None
    CLASSES = ([names[i] for i in sorted(names)] if isinstance(names, dict) else list(names or []))
    if not CLASSES:
        seen = {int(float(line.split()[0]))
                for _, lbl in SPLITS["train"] if lbl and lbl.exists()
                for line in lbl.read_text().strip().splitlines() if line.split()}
        n = max(seen) + 1
        CLASSES = PCB_DEFECT_NAMES[:n] if n <= len(PCB_DEFECT_NAMES) else [str(i) for i in range(n)]
        print(f"\n⚠️ data.yaml에 클래스 정의가 없어 라벨에서 {n}종을 역산했습니다: {CLASSES}")
    print(f"\n박스 출처: 데이터셋 정답(GT) · 클래스 {len(CLASSES)}종: {CLASSES}")

print(f"train {len(SPLITS['train']):,}장 · valid {len(SPLITS['valid']):,}장 처리 예정")

In [ ]:
# 우회 경로에서만 실행됨 — 학습된 best.pt 업로드 (로컬 backend/yolo/weights/best.pt)
#   클래스 이름도 이 체크포인트에서 가져오므로 data.yaml이 없어도 된다.
DET = None
if BOX_SOURCE == "model":
    from google.colab import files
    from ultralytics import YOLO

    print("backend/yolo/weights/best.pt 를 선택하세요.")
    DET = YOLO(next(iter(files.upload())))
    CLASSES = [DET.names[i] for i in sorted(DET.names)]
    print(f"\n박스 출처: best.pt 예측 · 클래스 {len(CLASSES)}종: {CLASSES}")
else:
    print("정답 라벨이 있으므로 이 셀은 건너뜁니다.")

## 3. 자동 폴리곤 라벨링 (핵심 단계)

각 이미지의 GT bbox를 MobileSAM에 프롬프트로 넣어 마스크를 얻고,
윤곽선을 추출해 YOLO 세그멘테이션 포맷(`class x1 y1 x2 y2 ... xn yn`, 정규화 좌표)으로 저장한다.

**품질 게이트** — 아래 조건에 걸리면 해당 인스턴스는 폴리곤 대신 원본 사각형으로 대체한다.
SAM이 실패한 케이스를 조용히 흘려보내지 않기 위한 장치다.
- 마스크가 비었거나 너무 작음 (bbox 면적의 5% 미만)
- 마스크가 bbox 밖으로 크게 번짐 (bbox 면적의 150% 초과)
- 윤곽점이 3개 미만

T4 기준 20~40분 소요.

In [ ]:
import shutil

import cv2
import numpy as np
import torch
from ultralytics import SAM

sam = SAM("mobile_sam.pt")

DST = Path("/content/pcb_seg")
MIN_RATIO, MAX_RATIO = 0.05, 1.5  # 마스크/bbox 면적비 허용 범위
DET_CONF = 0.25                   # 우회 경로에서 예측 박스를 채택할 최소 확신도

stats = {"polygon": 0, "fallback": 0, "images": 0, "skipped": 0}


def mask_to_polygon(mask, w, h):
    """이진 마스크에서 최대 윤곽을 뽑아 정규화 폴리곤 좌표로 변환한다."""
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    # 점 수를 줄여 라벨 파일 크기와 학습 부하를 낮춘다 (둘레의 0.5%를 허용 오차로)
    c = cv2.approxPolyDP(c, 0.005 * cv2.arcLength(c, True), True).reshape(-1, 2)
    if len(c) < 3:
        return None
    return [(x / w, y / h) for x, y in c]


def read_boxes(img, img_path, lbl_path):
    """(클래스, xyxy 절대좌표) 목록을 만든다. 정답 라벨이 있으면 그것을, 없으면 모델 예측을 쓴다."""
    h, w = img.shape[:2]
    if BOX_SOURCE == "labels":
        out = []
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            cls = int(float(parts[0]))
            cx, cy, bw, bh = (float(v) for v in parts[1:5])
            out.append((cls, [(cx - bw / 2) * w, (cy - bh / 2) * h,
                              (cx + bw / 2) * w, (cy + bh / 2) * h]))
        return out
    r = DET.predict(img, conf=DET_CONF, verbose=False)[0]
    return [(int(b.cls), b.xyxy[0].tolist()) for b in r.boxes]


for split, pairs in SPLITS.items():
    out_img, out_lbl = DST / split / "images", DST / split / "labels"
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    for i, (img_path, lbl_path) in enumerate(pairs):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        boxes = read_boxes(img, img_path, lbl_path)
        if not boxes:
            stats["skipped"] += 1  # 결함이 없는 이미지 — 배경 샘플로 이미지만 복사한다
            (out_lbl / f"{img_path.stem}.txt").write_text("")
            shutil.copy(img_path, out_img / img_path.name)
            stats["images"] += 1
            continue

        prompts = torch.tensor([b for _, b in boxes], dtype=torch.float32)
        res = sam(img, bboxes=prompts.clone(), verbose=False)[0]  # SAM이 입력을 변형하므로 복사본 전달
        masks = res.masks.data.cpu().numpy() if res.masks is not None else []

        lines = []
        for (cls, box), mk in zip(boxes, masks):
            m8 = (mk > 0.5).astype(np.uint8)
            box_area = max(1.0, (box[2] - box[0]) * (box[3] - box[1]))
            poly = None
            if MIN_RATIO <= m8.sum() / box_area <= MAX_RATIO:
                poly = mask_to_polygon(m8, w, h)
            if poly is None:  # SAM 실패 — 사각형을 4점 폴리곤으로 대체
                x1, y1, x2, y2 = box
                poly = [(x1 / w, y1 / h), (x2 / w, y1 / h), (x2 / w, y2 / h), (x1 / w, y2 / h)]
                stats["fallback"] += 1
            else:
                stats["polygon"] += 1
            lines.append(f"{cls} " + " ".join(f"{x:.6f} {y:.6f}" for x, y in poly))

        (out_lbl / f"{img_path.stem}.txt").write_text("\n".join(lines))
        shutil.copy(img_path, out_img / img_path.name)
        stats["images"] += 1

        if (i + 1) % 500 == 0:
            print(f"  {split}: {i + 1:,}/{len(pairs):,}")
    print(f"{split} 완료 — 누적 {stats['images']:,}장")

total = stats["polygon"] + stats["fallback"]
assert total, "인스턴스를 하나도 만들지 못했습니다 — 박스 출처를 확인하세요."
print(f"\n이미지 {stats['images']:,}장 (결함 없음 {stats['skipped']:,}장) · 인스턴스 {total:,}개")
print(f"폴리곤 생성 {stats['polygon']:,} ({stats['polygon'] / total:.1%}) · "
      f"사각형 대체 {stats['fallback']:,} ({stats['fallback'] / total:.1%})")
print("\n→ 폴리곤 생성률이 90% 아래면 다음 셀의 검수 결과를 특히 꼼꼼히 볼 것")

## 4. 표본 검수 (반드시 눈으로 확인할 것)

자동 생성 라벨을 그대로 믿지 않는다. 무작위 표본의 폴리곤이 실제 결함 형상을 따라가는지 확인한다.
**빨간 윤곽이 결함이 아닌 배경을 감싸고 있으면 학습으로 넘어가지 말 것.**

In [ ]:
import random

import matplotlib.pyplot as plt

# 결함이 있는 이미지만 골라 검수한다 (빈 라벨은 볼 것이 없다)
pool = [p for p in sorted((DST / "train" / "images").iterdir())
        if (DST / "train" / "labels" / f"{p.stem}.txt").stat().st_size > 0]
print(f"검수 대상 {len(pool):,}장 중 무작위 9장")
samples = random.sample(pool, min(9, len(pool)))
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for ax, img_path in zip(axes.ravel(), samples):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    label = (DST / "train" / "labels" / f"{img_path.stem}.txt").read_text().strip()

    box = None
    for line in label.splitlines():
        parts = line.split()
        cls, vals = int(parts[0]), [float(v) for v in parts[1:]]
        pts = np.array([(vals[i] * w, vals[i + 1] * h) for i in range(0, len(vals), 2)], np.int32)
        cv2.polylines(img, [pts], True, (255, 0, 0), 2)
        cv2.putText(img, CLASSES[cls], tuple(pts[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
        if box is None:  # 첫 인스턴스 주변으로 확대해 형상을 확인한다
            pad = 70
            box = (max(0, pts[:, 0].min() - pad), max(0, pts[:, 1].min() - pad),
                   min(w, pts[:, 0].max() + pad), min(h, pts[:, 1].max() + pad))

    ax.imshow(img[box[1]:box[3], box[0]:box[2]] if box else img)
    ax.set_title(img_path.stem[:24], fontsize=9)
    ax.axis("off")

for ax in axes.ravel()[len(samples):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 5. 세그멘테이션용 data.yaml 생성
SEG_YAML = DST / "data.yaml"
SEG_YAML.write_text(
    f"path: {DST}\n"
    "train: train/images\n"
    "val: valid/images\n\n"
    f"nc: {len(CLASSES)}\n"
    f"names: {CLASSES}\n"
)
print(SEG_YAML.read_text())

In [ ]:
# 6. 세그멘테이션 모델 학습 (T4 기준 약 1.5~2시간)
#    검출 모델과 동일한 하이퍼파라미터를 써서 두 접근을 공정하게 비교한다.
from ultralytics import YOLO

seg = YOLO("yolo11n-seg.pt")
seg.train(
    data=str(SEG_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    project="runs",
    name="pcb_seg",
)

In [ ]:
# 7. 성능 확인 — 이 출력을 그대로 복사해 두면 포트폴리오 지표로 쓸 수 있다
metrics = seg.val()

print(f"Box  mAP50: {metrics.box.map50:.4f}   mAP50-95: {metrics.box.map:.4f}")
print(f"Mask mAP50: {metrics.seg.map50:.4f}   mAP50-95: {metrics.seg.map:.4f}")
print(f"Mask Precision: {metrics.seg.mp:.4f}   Recall: {metrics.seg.mr:.4f}\n")

print("클래스별 Mask mAP50")
for i, name in metrics.names.items():
    print(f"  {name:18} {metrics.seg.ap50[i]:.4f}")

In [ ]:
# 8. 결함 면적 계측 데모 — 세그멘테이션으로 새로 가능해진 것
#    bbox로는 얻을 수 없던 실제 결함 픽셀 면적을 산출한다.
shown = 0
for sample in sorted((DST / "valid" / "images").iterdir()):
    r = seg.predict(str(sample), conf=0.25, verbose=False)[0]
    if r.masks is None or len(r.boxes) == 0:
        continue
    print(sample.name)
    for box, poly in zip(r.boxes, r.masks.xy):
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        bbox_px = (x2 - x1) * (y2 - y1)
        mask_px = cv2.contourArea(poly.astype(np.float32))
        print(f"  {r.names[int(box.cls)]:18} 결함면적 {mask_px:8.0f}px   "
              f"bbox {bbox_px:8.0f}px   충전율 {mask_px / bbox_px:6.1%}")
    shown += 1
    if shown == 3:
        break

In [ ]:
# 9. 결과물 다운로드
#    best.pt → 로컬 backend/yolo/weights/best-seg.pt 로 저장할 것 (기존 best.pt를 덮어쓰지 말 것)
#    플롯·results.csv → frontend/public/model/ 로 저장 (모델 성능 탭 갱신용)
#
#    Ultralytics의 runs_dir 설정에 따라 저장 경로가 달라지므로 경로를 추측하지 않고 직접 찾는다.
from google.colab import files

cands = sorted(Path("/content").rglob("weights/best.pt"),
               key=lambda p: p.stat().st_mtime, reverse=True)
for p in cands:
    print(f"{p}  {p.stat().st_size / 1e6:.1f} MB")

if not cands:
    raise SystemExit("best.pt를 찾지 못했습니다 — 학습이 완료되었는지 확인하세요.")

best = cands[0]
run_dir = best.parent.parent
print(f"\n다운로드: {best}")
files.download(str(best))
for f in ["results.csv", "MaskPR_curve.png", "BoxPR_curve.png",
          "confusion_matrix_normalized.png", "results.png"]:
    if (run_dir / f).exists():
        files.download(str(run_dir / f))